In [50]:
%pip install natsort
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares
from IPython.display import display
import matplotlib.pyplot as plt
import plotly.express as px
import numpy as np
import re
from natsort import natsorted

# =============================================================================
# 1. 参数区
# =============================================================================
folder_path = Path(r"D:\毕设数据\20_export_pulse\20_export_pulse\METABatt_Sony_Murata_18650VTC6_007")
SOH_FILENAME_PATTERN = re.compile(r"BM\d+_(\d+(?:\.\d+)?)SOH\.parquet$", flags=re.IGNORECASE)


def extract_soh_from_filename(filename):
    match = SOH_FILENAME_PATTERN.search(filename.strip())
    if match is None:
        raise ValueError(f"无法从文件名解析 SOH: {filename}")
    return float(match.group(1))

# 实验设置的SOC顺序
SOC_ORDER = ["90%", "50%", "10%"]

# cycle内脉冲按照ID划分
REMOVE_PULSE_BEFORE_MIN = 60

# 实测 active 会比 3h 稍大，但不会超过 CYCLE_ACTIVE_LIMIT_HOUR.
# 因此用 CYCLE_ACTIVE_LIMIT_HOUR 作为 time_diff 判断是否进入下一 cycle 的边界。
CYCLE_ACTIVE_LIMIT_HOUR = 4.0

# 设置电流标准差
STD_LIMIT_1P5A = 0.1
STD_LIMIT_3A = 0.1

OUTPUT_COLUMNS = [
    "SOH",
    "SOC",
    "File",
    "Time",
    "Current",
    "Zustand",
    "ID",
    "Zustand/Current",
]

Note: you may need to restart the kernel to use updated packages.


In [51]:
# =============================================================================
# 2. 读取 parquet
# =============================================================================

parquet_files = natsorted(list(folder_path.rglob("*.parquet")))

if len(parquet_files) == 0:
    raise FileNotFoundError(f"没有在文件夹中找到 parquet 文件: {folder_path}")

df_list = []

for file in parquet_files:
    temp = pd.read_parquet(file)
    temp["File"] = file.name
    temp["SOH"] = extract_soh_from_filename(file.name)

    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

In [52]:
# =============================================================================
# 3. time diff 主函数
# =============================================================================

def build_time_diff_sequence(df):
    df_td = df.copy()

    # -----------------------------
    # 1. 基础时间与电流处理
    # -----------------------------
    df_td["Time"] = pd.to_datetime(df_td["Time"], utc=True, errors="coerce")
    df_td["Current"] = pd.to_numeric(df_td["Current"], errors="coerce")

    df_td = df_td.dropna(subset=["Time", "Current"]).copy()
    df_td = df_td.sort_values(["File", "Time"]).reset_index(drop=True)

    # -----------------------------
    # 2. 按 time_diff 划分 cycle / SOC
    # -----------------------------
    # 直接比较同一个 File 内相邻时间点的时间差：
    #   time_diff <= CYCLE_ACTIVE_LIMIT_HOUR：仍属于当前 cycle
    #   time_diff >  CYCLE_ACTIVE_LIMIT_HOUR：说明中间经过 pause，进入下一个 cycle
    df_td["time_diff_hour"] = (df_td.groupby("File")["Time"].diff()/ pd.Timedelta(hours=1))
    
    df_td["is_new_cycle"] = (
        df_td["time_diff_hour"].isna()
        | (df_td["time_diff_hour"] > CYCLE_ACTIVE_LIMIT_HOUR)
    )

    df_td["cycle_id"] = (
        df_td.groupby("File")["is_new_cycle"]
        .cumsum()
        .astype(int)
    )

    df_td["SOC"] = df_td["cycle_id"].map(
        lambda cycle_id: SOC_ORDER[(cycle_id - 1) % len(SOC_ORDER)]
    )

    cycle_start_time = df_td.groupby(["File", "cycle_id"])["Time"].transform("min")

    df_td["time_from_cycle_start_min"] = (df_td["Time"] - cycle_start_time) / pd.Timedelta(minutes=1)

    df_td["Current"] = df_td["Current"]

    # -----------------------------
    # 3. 统一 Zustand
    # -----------------------------
    df_td["Zustand"] = df_td["Zustand"].astype(str)

    df_td.loc[
        df_td["Zustand"].str.startswith("DCH", na=False),
        "Zustand"
    ] = "DCH"

    df_td.loc[
        df_td["Zustand"].str.startswith("CHA", na=False),
        "Zustand"
    ] = "CHA"

    # -----------------------------
    # 4. 生成 pulse_segment_id
    # -----------------------------
    df_td["pulse_segment_id"] = (
        df_td["File"].ne(df_td["File"].shift())
        | df_td["Zustand"].ne(df_td["Zustand"].shift())
    ).cumsum()

    # -----------------------------
    # 5. 只保留 CHA / DCH
    # -----------------------------
    pulse_mask = df_td["Zustand"].str.startswith(("CHA", "DCH"), na=False)
    
    df_td["Zustand/Current"] = df_td["Zustand"] + "/" + (df_td["Current"].astype(float).round(1)).astype(str)

    pulse_sequence = df_td[pulse_mask].copy()
    pulse_sequence = pulse_sequence.sort_values(
        ["File", "pulse_segment_id", "Time"]
    )

    # -----------------------------
    # 6. 剔除电流不稳定的 pulse_segment_id
    # -----------------------------
    def is_bad_current_segment(group):
        group = group.sort_values("Time")

        current_values = group["Current"]

        # 如果首个测量点 Current == 0，则忽略首点
        if len(group) >= 2 and group["Current"].iloc[0] == 0:
            current_values = current_values.iloc[1:]

        current_abs_level = round(current_values.abs().iloc[0], 1)
        current_std = current_values.std()

        if current_abs_level == 1.5:
            return current_std > STD_LIMIT_1P5A

        if current_abs_level == 3.0:
            return current_std > STD_LIMIT_3A

        return True
    
    pulse_sequence = pulse_sequence.groupby(["File", "pulse_segment_id"]).filter(
    lambda group: not is_bad_current_segment(group)).copy()

    # -----------------------------
    # 7. 每个 pulse_segment_id 只取一个代表点
    # -----------------------------
    def pick_pulse_row(group):
        group = group.sort_values("Time")

        # 如果该脉冲段第一个测量点 Current
        # 就改用第二个测量点记录
        if group["Current"].iloc[0] == 0 and len(group) >= 2:
            return group.iloc[1]

        return group.iloc[0]

    pulse_sequence = (
        pulse_sequence
        .groupby(["File", "pulse_segment_id"], group_keys=False)
        .apply(pick_pulse_row)
        .reset_index(drop=True)
    )

    # -----------------------------
    # 8. 只保留最终输出列
    # -----------------------------
    
    
    pulse_sequence = pulse_sequence[OUTPUT_COLUMNS].copy()

    return pulse_sequence

pulse_sequence = build_time_diff_sequence(df)

display(pulse_sequence)

C:\Users\12639\AppData\Local\Temp\ipykernel_32420\1104026746.py:123: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(pick_pulse_row)


,SOH,SOC,File,Time,Current,Zustand,ID,Zustand/Current
0,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 12:03:52.500000+00:00,-1.498660,DCH,12_17,DCH/-1.5
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,DCH,12_25,DCH/-3.0
3,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 19:55:56.400000+00:00,-1.497221,DCH,12_35,DCH/-1.5
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,CHA,12_39,CHA/1.5
...,...,...,...,...,...,...,...,...
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,CHA,9_48,CHA/3.0
258,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 04:22:48.550000+00:00,-1.494522,DCH,9_54,DCH/-1.5
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,DCH,9_62,DCH/-3.0


In [ ]:
# R0 计算部分
# -----------------------------
# 9. 筛选脉冲/清除1.5A下的DCH脉冲
 # -----------------------------
def filter_pulse(df):
    pulse_sequence_filter = pulse_sequence[~pulse_sequence["Zustand/Current"].isin(["DCH/-1.5"])]
    return pulse_sequence_filter
display(filter_pulse(pulse_sequence))

,SOH,SOC,File,Time,Current,Zustand,ID,Zustand/Current
1,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 13:04:35.510000+00:00,1.498165,CHA,12_21,CHA/1.5
2,90.3,90%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 14:05:18.700000+00:00,-2.995366,DCH,12_25,DCH/-3.0
4,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 20:56:39.720000+00:00,1.499155,CHA,12_39,CHA/1.5
5,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 21:57:23.020000+00:00,-2.997524,DCH,12_43,DCH/-3.0
6,90.3,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM12_...,2024-11-12 22:58:26.330000+00:00,2.992477,CHA,12_47,CHA/3.0
...,...,...,...,...,...,...,...,...
256,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 22:29:49.320000+00:00,-2.998784,DCH,9_44,DCH/-3.0
257,92.2,50%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-23 23:30:52.610000+00:00,2.999132,CHA,9_48,CHA/3.0
259,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 05:23:31.800000+00:00,1.496366,CHA,9_58,CHA/1.5
260,92.2,10%,METABatt_Sony_Murata_18650VTC6_007_pulse_BM9_9...,2024-10-24 06:24:14.960000+00:00,-2.988168,DCH,9_62,DCH/-3.0


In [ ]:
pause_end = df_td["Zustand"]